In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import gc
import time
#import matplotlib.pyplot as plt
#from livelossplot.keras import PlotLossesCallback
from tensorflow.keras.callbacks import ReduceLROnPlateau
from keras.models import load_model

Using TensorFlow backend.


In [99]:
data_train=pd.read_csv('traindata20200520.csv',header=0,encoding='gbk')
data_train

,Vegetation,QH_DEM,QH_Aspect,QH_Slope,GPP2019169,GPP2019177,GPP2019185,GPP2019193,GPP2019201,GPP2019209,...,SR20192575,SR20192576,SR20192577,SR20192651,SR20192652,SR20192653,SR20192654,SR20192655,SR20192656,SR20192657
0,小嵩草、垂穗披碱草草甸,3740,251.813995,31.240200,250,81,293,229,327,279,...,2452,1585,761,761,2471,379,677,3086,2213,1100
1,小嵩草、垂穗披碱草草甸,3740,251.813995,31.240200,250,81,293,229,327,279,...,2452,1585,761,761,2471,379,677,3086,2213,1100
2,小嵩草、垂穗披碱草草甸,3755,289.044006,24.078699,239,81,298,223,328,279,...,2484,1637,821,772,2445,391,684,3092,2226,1107
3,小嵩草、垂穗披碱草草甸,3755,289.044006,24.078699,239,81,298,223,328,279,...,2484,1637,821,772,2445,391,684,3092,2226,1107
4,小嵩草、垂穗披碱草草甸,3785,275.941986,28.350700,246,81,295,230,327,278,...,2511,1651,836,766,2435,391,683,3083,2213,1098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2899,西北针茅草原,4193,135.093002,3.394220,109,40,139,113,160,127,...,4587,3712,2496,1473,2760,716,1152,3664,3333,2339
2900,西北针茅草原,4200,233.194000,9.060290,122,41,144,116,173,134,...,4596,3734,2471,1417,2745,694,1113,3635,3259,2211
2901,西北针茅草原,4200,233.194000,9.060290,122,41,144,116,173,134,...,4596,3734,2471,1417,2745,694,1113,3635,3259,2211
2902,西北针茅草原,4193,310.510010,4.988270,133,44,149,120,185,140,...,4509,3645,2401,1415,2747,692,1112,3636,3261,2211


In [68]:
data_train.info()
data_train.Vegetation.unique()
#len(data_train.Vegetation.unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2904 entries, 0 to 2903
Columns: 163 entries, Vegetation to SR20192657
dtypes: float64(6), int64(156), object(1)
memory usage: 3.6+ MB


array(['小嵩草、垂穗披碱草草甸', '线叶嵩草、珠芽蓼、圆穗蓼草甸', '金露梅灌丛', '线叶嵩草草甸', '小嵩草草甸',
       '垫状金露梅灌丛', '藏嵩草沼泽化草甸', '高山嵩草草甸', '嵩草草甸', '垂穗披碱草草甸', '芨芨草草甸',
       '高山嵩草、异针茅草原化草甸', '矮生嵩草草甸', '冰草、紫花针茅草原', '线叶嵩草-紫花针茅草甸', '紫花针茅草原',
       '短花针茅草原', '藏北嵩草沼泽草甸', '芨芨草草原', '赖草草甸', '长芒草、赖草、猪毛蒿草原', '沙生针茅草原',
       '矮生嵩草、紫花针茅草甸', '华扁穗草沼泽草甸', '矮生嵩草、紫花针茅草原化草甸', '藏嵩草沼泽草甸',
       '小嵩草、黄帚橐吾草甸', '小嵩草、青海刺参、黄帚橐吾草甸', '鹅绒委陵菜、青海刺参草甸', '黄帚橐吾、圆穗蓼草甸',
       '珠芽蓼、圆穗蓼草甸', '小嵩草、珠芽蓼、圆穗蓼草甸', '矮嵩草、垂穗披碱草草甸', '小嵩草、黄帚橐吾、圆穗蓼草甸',
       '小嵩草、白苞筋骨草草甸', '高山流石坡稀疏植被', '青藏苔草、藏嵩草草原', '早熟禾、细叶西伯利亚蓼、海乳草草原',
       '小嵩草、早熟禾草原化草甸', '披针叶黄华、细叶亚菊草原', '冷蒿草原', '匍匐水柏枝与嵩草草甸复合体',
       '小嵩草、矮嵩草草甸', '藏西风毛菊、细叶亚菊草原', '异针茅草原', '异针茅、垫状点地梅草原', '异针茅、青海早熟禾草原',
       '异针茅、臭蒿草原', '长芒草草原', '小嵩草、异针茅草原化草甸', '三芒草草甸', '山生柳灌丛', '苔草草甸',
       '细叶亚菊草原化草甸', '疏花针茅草原', '青海早熟禾、扇穗茅草甸', '驼绒藜荒漠', '披碱草草甸', '西藏嵩草沼泽草甸',
       '线叶嵩草、紫花针茅草原化草甸', '高山草甸', '蒿叶猪毛菜砾漠', '矮生嵩草', '高山苔草草原', '苔草、藏嵩草草甸',
       '头花杜鹃灌丛', '鹅绒委陵菜草甸', '冰岛蓼草甸', '黑穗苔草草甸', '小嵩草-紫花针茅草原化草甸', '重齿风毛菊草甸',
       '小嵩草草原化

In [105]:
pd.set_option('display.max_rows', None)
print(data_train['Vegetation'].value_counts())

矮生嵩草草甸                   355
小嵩草草甸                    274
高山嵩草草甸                   182
长芒草、赖草、猪毛蒿草原             177
垂穗披碱草草甸                  145
芨芨草草原                    124
线叶嵩草草甸                   120
紫花针茅草原                    89
金露梅灌丛                     78
冷蒿草原                      70
异针茅草原                     69
线叶嵩草、珠芽蓼、圆穗蓼草甸            64
小嵩草-紫花针茅草原化草甸             47
青海早熟禾、扇穗茅草甸               46
小嵩草、青海刺参、黄帚橐吾草甸           44
藏嵩草沼泽化草甸                  41
青海早熟禾草甸                   41
西北针茅草原                    40
青藏苔草草原                    40
矮生嵩草、紫花针茅草原化草甸            33
嵩草草甸                      30
小嵩草、西北针茅草原化草甸             30
青藏苔草、矮嵩草草甸                30
矮嵩草草甸                     30
小嵩草、矮嵩草草甸                 30
芨芨草草甸                     29
华扁穗草沼泽草甸                  27
垫状金露梅灌丛                   26
矮生嵩草、紫花针茅草甸               24
赖草草甸                      21
小嵩草、早熟禾草原化草甸              20
珠芽蓼、圆穗蓼草甸                 20
鹅绒委陵菜草甸                   20
藏嵩草沼泽草甸                   20
小嵩草、紫花针茅草原化草甸 

### 顺序编码

In [374]:
# 将其映射到0,1,2...上
class_dict = {'swamp_and_hydrophytic_vegetation':0, 'alpine_steppe':2,
       'alpine_meadow':1, 'temperate_steppe':2, 'meadow':1, 'salt_marsh/swamp':0,
       'semi_shrubby_desert':3, 'shrubby_desert':3, 'dwarf_tree_desert':3,
       'cushion_vegetation':4}
data_train["Vegetation"] = data_train.Vegetation.map(class_dict)
data_train.head()

,Vegetation,DEM,ASPECT,SLOPE,GPP2019169,GPP2019177,GPP2019185,GPP2019193,GPP2019201,GPP2019209,...,PSN2019193,PSN2019201,PSN2019209,PSN2019217,PSN2019225,PSN2019233,PSN2019241,PSN2019249,PSN2019257,PSN2019265
0,0,4298,69.232430,15.326968,208.0,150.0,264.0,248.0,337.0,287.0,...,29787.0,29787.0,29780.0,29780.0,29787.0,29774.0,29783.0,29783.0,29780.0,29782.0
1,2,3403,225.711334,2.215723,87.0,92.0,90.0,110.0,122.0,143.0,...,82.0,99.0,98.0,95.0,72.0,68.0,63.0,58.0,2.0,34.0
2,2,3378,137.565613,2.933709,136.0,138.0,127.0,149.0,169.0,185.0,...,126.0,153.0,170.0,139.0,121.0,99.0,106.0,80.0,3.0,39.0
3,2,3546,210.834885,5.390974,123.0,129.0,144.0,160.0,221.0,193.0,...,147.0,203.0,175.0,142.0,146.0,122.0,102.0,85.0,2.0,50.0
4,0,3767,212.595642,3.668648,112.0,174.0,192.0,212.0,277.0,307.0,...,222.0,259.0,265.0,215.0,218.0,144.0,182.0,114.0,3.0,79.0


### 独热编码

In [69]:
data_train = data_train.join(pd.get_dummies(data_train.Vegetation))
#del data_train["Vegetation"]
data_train.head()

,Vegetation,QH_DEM,QH_Aspect,QH_Slope,GPP2019169,GPP2019177,GPP2019185,GPP2019193,GPP2019201,GPP2019209,...,驼绒藜荒漠,高山嵩草、异针茅草原化草甸,高山嵩草草甸,高山流石坡稀疏植被,高山苔草草原,高山草甸,鹅绒委陵菜、青海刺参草甸,鹅绒委陵菜草甸,黄帚橐吾、圆穗蓼草甸,黑穗苔草草甸
0,小嵩草、垂穗披碱草草甸,3740,251.813995,31.240200,250,81,293,229,327,279,...,0,0,0,0,0,0,0,0,0,0
1,小嵩草、垂穗披碱草草甸,3740,251.813995,31.240200,250,81,293,229,327,279,...,0,0,0,0,0,0,0,0,0,0
2,小嵩草、垂穗披碱草草甸,3755,289.044006,24.078699,239,81,298,223,328,279,...,0,0,0,0,0,0,0,0,0,0
3,小嵩草、垂穗披碱草草甸,3755,289.044006,24.078699,239,81,298,223,328,279,...,0,0,0,0,0,0,0,0,0,0
4,小嵩草、垂穗披碱草草甸,3785,275.941986,28.350700,246,81,295,230,327,278,...,0,0,0,0,0,0,0,0,0,0


### 打乱顺序

In [70]:
# 生成该区间的随意唯一索引
index = np.random.permutation(len(data_train))
# 用生成的乱的索引就能将其打乱了
data_train = data_train.iloc[index ,:]

### 顺序编码的标签

In [376]:
train_x = data_train.iloc[:,1:]
train_y = data_train.loc[:, ['Vegetation']]  
train_x.head(), train_y.head()

(       DEM      ASPECT      SLOPE  GPP2019169  GPP2019177  GPP2019185  \
 514   3377  308.916992   0.490046       163.0        56.0       340.0   
 249   3242  146.961166   8.436536       495.0       391.0       528.0   
 68    3559  226.310974   8.014600        94.0       265.0       224.0   
 764   4529  355.621155   4.955467        28.0        24.0        35.0   
 1160  4362   52.506660  20.456871       216.0       211.0       276.0   
 
       GPP2019193  GPP2019201  GPP2019209  GPP2019217  ...  PSN2019193  \
 514        372.0       441.0       361.0       358.0  ...       303.0   
 249        622.0       767.0       632.0       671.0  ...       470.0   
 68         357.0       311.0       262.0       362.0  ...       309.0   
 764         24.0        40.0        42.0        51.0  ...      4179.0   
 1160       256.0       366.0       289.0       316.0  ...       204.0   
 
       PSN2019201  PSN2019209  PSN2019217  PSN2019225  PSN2019233  PSN2019241  \
 514        375.0       331

### 独热编码的标签

In [74]:
train_x = data_train.iloc[:, 1:-86]
train_x.head()

,QH_DEM,QH_Aspect,QH_Slope,GPP2019169,GPP2019177,GPP2019185,GPP2019193,GPP2019201,GPP2019209,GPP2019217,...,SR20192575,SR20192576,SR20192577,SR20192651,SR20192652,SR20192653,SR20192654,SR20192655,SR20192656,SR20192657
1079,4615,86.630699,4.897850,146,141,180,102,234,214,225,...,2636,1826,848,870,2265,397,700,3100,2331,1293
1337,4094,87.838699,3.424770,57,18,139,122,170,135,122,...,2625,2266,1447,1333,2411,724,1092,3155,2775,1895
2415,4191,80.212601,46.724602,130,175,184,80,223,207,185,...,4350,3475,1934,1143,2699,536,926,3723,3276,1991
2871,4102,138.281006,7.053000,108,11,95,102,128,104,98,...,2861,2282,1624,1600,2907,818,1281,3719,3408,2547
302,3714,245.317993,10.040000,444,106,335,318,388,363,364,...,3297,2906,1522,940,2648,451,799,3734,3281,1879


In [75]:
train_y = data_train.iloc[:, -86:]
train_y.head()

,三芒草草甸,冰岛蓼草甸,冰草、垂穗披碱草草原,冰草、紫花针茅草原,冷蒿草原,匍匐水柏枝与嵩草草甸复合体,华扁穗草沼泽草甸,垂穗披碱草草甸,垫状金露梅灌丛,头花杜鹃灌丛,...,驼绒藜荒漠,高山嵩草、异针茅草原化草甸,高山嵩草草甸,高山流石坡稀疏植被,高山苔草草原,高山草甸,鹅绒委陵菜、青海刺参草甸,鹅绒委陵菜草甸,黄帚橐吾、圆穗蓼草甸,黑穗苔草草甸
1079,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1337,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2415,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2871,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
302,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [97]:
print(np.argmax(train_y, axis=1))

ValueError: Shape of passed values is (2904, 1), indices imply (2904, 86)

In [117]:
def multi_category_focal_loss1(alpha, gamma=2.0):
    """
    focal loss for multi category of multi label problem
    适用于多分类或多标签问题的focal loss
    alpha用于指定不同类别/标签的权重，数组大小需要与类别个数一致
    当你的数据集不同类别/标签之间存在偏斜，可以尝试适用本函数作为loss
    Usage:
     model.compile(loss=[multi_category_focal_loss1(alpha=[1,2,3,2], gamma=2)], metrics=["accuracy"], optimizer='adam')
    """
    epsilon = 1.e-7
    alpha = tf.constant(alpha, dtype=tf.float32)
    #alpha = tf.constant([[1],[1],[1],[1],[1]], dtype=tf.float32)
    #alpha = tf.constant_initializer(alpha)
    gamma = float(gamma)
    def multi_category_focal_loss1_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        y_t = tf.multiply(y_true, y_pred) + tf.multiply(1-y_true, 1-y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.math.pow(tf.subtract(1., y_t), gamma)
        fl = tf.matmul(tf.multiply(weight, ce), alpha)
        loss = tf.reduce_mean(fl)
        return loss
    return multi_category_focal_loss1_fixed

In [236]:
def multi_category_focal_loss2(gamma=2., alpha=.25):
    """
    focal loss for multi category of multi label problem
    适用于多分类或多标签问题的focal loss
    alpha控制真值y_true为1/0时的权重
        1的权重为alpha, 0的权重为1-alpha
    当你的模型欠拟合，学习存在困难时，可以尝试适用本函数作为loss
    当模型过于激进(无论何时总是倾向于预测出1),尝试将alpha调小
    当模型过于惰性(无论何时总是倾向于预测出0,或是某一个固定的常数,说明没有学到有效特征)
        尝试将alpha调大,鼓励模型进行预测出1。
    Usage:
     model.compile(loss=[multi_category_focal_loss2(alpha=0.25, gamma=2)], metrics=["accuracy"], optimizer='adam')
    """
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)

    def multi_category_focal_loss2_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
        alpha_t = y_true*alpha + (tf.ones_like(y_true)-y_true)*(1-alpha)
        y_t = tf.multiply(y_true, y_pred) + tf.multiply(1-y_true, 1-y_pred)
        y_t = tf.clip_by_value(y_t, epsilon, 1. - epsilon)#这句是后加的
        ce = -tf.math.log(y_t)
        weight = tf.math.pow(tf.subtract(1., y_t), gamma)
        fl = tf.multiply(tf.multiply(weight, ce), alpha_t)
        loss = tf.reduce_mean(fl)
        return loss
    return multi_category_focal_loss2_fixed

In [68]:
def focal_loss(pred, y, alpha=0.25, gamma=2):
        r"""Compute focal loss for predictions.
            Multi-labels Focal loss formula:
                FL = -alpha * (z-p)^gamma * log(p) -(1-alpha) * p^gamma * log(1-p)
                     ,which alpha = 0.25, gamma = 2, p = sigmoid(x), z = target_tensor.
        Args:
         pred: A float tensor of shape [batch_size, num_anchors,
            num_classes] representing the predicted logits for each class
         y: A float tensor of shape [batch_size, num_anchors,
            num_classes] representing one-hot encoded classification targets
         alpha: A scalar tensor for focal loss alpha hyper-parameter
         gamma: A scalar tensor for focal loss gamma hyper-parameter
        Returns:
            loss: A (scalar) tensor representing the value of the loss function
        """
        zeros = tf.zeros_like(pred, dtype='float32')
        pred = tf.cast(pred, tf.float32)
        y = tf.cast(y, tf.float32)

        # For positive prediction, only need consider front part loss, back part is 0;
        # target_tensor > zeros <=> z=1, so positive coefficient = z - p.
        pos_p_sub = tf.where(y > zeros, y - pred, zeros) # positive sample 寻找正样本，并进行填充

        # For negative prediction, only need consider back part loss, front part is 0;
        # target_tensor > zeros <=> z=1, so negative coefficient = 0.
        neg_p_sub = tf.where(y > zeros, zeros, pred) # negative sample 寻找负样本，并进行填充
        per_entry_cross_ent = - alpha * (pos_p_sub ** gamma) * tf.math.log(tf.clip_by_value(pred, 1e-8, 1.0)) \
                              - (1 - alpha) * (neg_p_sub ** gamma) * tf.math.log(tf.clip_by_value(1.0 - pred, 1e-8, 1.0))

        return tf.reduce_sum(per_entry_cross_ent)

In [74]:
def focal_loss(y_true, y_pred):
    alpha, gamma = 0.25, 2
    y_pred = tf.clip_by_value(y_pred, 1e-8, 1 - 1e-8)
    loss =  - alpha * y_true * tf.math.log(y_pred) * (1 - y_pred)**gamma\
           - (1 - alpha) * (1 - y_true) * tf.math.log(1 - y_pred) * y_pred**gamma
    return loss

### 顺序编码正确的Focal loss

In [305]:
def focal_loss(y_true, y_pred, alpha=0.25, gamma=2):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-8, 1 - 1e-8)#避免log出现负值，从而避免loss：nan
    loss = tf.reduce_mean(- tf.reduce_sum(alpha * y_true * tf.math.pow(1-y_pred, gamma) * tf.math.log(y_pred), axis=[1]))
    return loss

In [308]:
def Focal_Loss(y_true, y_pred, alpha=0.25, gamma=2):
    """
    focal loss for multi-class classification
    fl(pt) = -alpha*(1-pt)^(gamma)*log(pt)
    :param y_true: ground truth one-hot vector shape of [batch_size, nb_class]
    :param y_pred: prediction after softmax shape of [batch_size, nb_class]
    :param alpha:
    :param gamma:
    :return:
    """
    # # parameters
    # alpha = 0.25
    # gamma = 2
    y_true = tf.cast(y_true, tf.float32)

    # To avoid divided by zero
    y_pred += tf.keras.backend.epsilon()

    # Cross entropy
    ce = -y_true * tf.math.log(y_pred)

    # Not necessary to multiply y_true(cause it will multiply with CE which has set unconcerned index to zero ),
    # but refer to the definition of p_t, we do it
    weight = tf.math.pow(1 - y_pred, gamma) * y_true

    # Now fl has a shape of [batch_size, nb_class]
    # alpha should be a step function as paper mentioned, but it doesn't matter like reason mentioned above
    # (CE has set unconcerned index to zero)
    #
    # alpha_step = tf.where(y_true, alpha*np.ones_like(y_true), 1-alpha*np.ones_like(y_true))
    fl = ce * weight * alpha

    # Both reduce_sum and reduce_max are ok
    reduce_fl = tf.keras.backend.max(fl, axis=-1)

    return reduce_fl


### 网络构建

In [91]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(1024,input_dim=162))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dense(512))
model.add(tf.keras.layers.Dense(256))
model.add(tf.keras.layers.Dense(128))
model.add(tf.keras.layers.Dropout(rate=0.5))
model.add(tf.keras.layers.Dense(86, activation='softmax'))
#model.summary()

In [20]:
label_shape=train_y.apply(lambda x:x.sum())
label_shape

三芒草草甸           10
冰岛蓼草甸            3
冰草、垂穗披碱草草原      10
冰草、紫花针茅草原        9
冷蒿草原            70
                ..
高山草甸            10
鹅绒委陵菜、青海刺参草甸     6
鹅绒委陵菜草甸         20
黄帚橐吾、圆穗蓼草甸      10
黑穗苔草草甸           7
Length: 86, dtype: int64

In [402]:
label_shape=[[762],[126],[32],[14],[120],[33],[20],[2],[2],[78]]

In [40]:
# 让学习率自动变化
reduce_lr = ReduceLROnPlateau(monitor='loss',factor=0.1, patience=20, mode='auto')

In [92]:
model.compile(loss='categorical_crossentropy', metrics=['acc'], optimizer='adam')
#顺序编码用loss='sparse_categorical_crossentropy'
#独热编码用loss='categorical_crossentropy'
model.summary()

Model: "sequential_16"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_119 (Dense)            (None, 1024)              166912    
_________________________________________________________________
batch_normalization_15 (Batc (None, 1024)              4096      
_________________________________________________________________
dense_120 (Dense)            (None, 512)               524800    
_________________________________________________________________
dense_121 (Dense)            (None, 256)               131328    
_________________________________________________________________
dense_122 (Dense)            (None, 128)               32896     
_________________________________________________________________
dropout_15 (Dropout)         (None, 128)               0         
_________________________________________________________________
dense_123 (Dense)            (None, 86)              

### 开始训练，e = epochs

In [93]:
e = 200
start = time.perf_counter()#time.process_time()测的时间偏长
history = model.fit(train_x,train_y, epochs=e,batch_size=4,verbose=1,callbacks=[reduce_lr])#
mins, secs = divmod((time.perf_counter() - start), 60)
hours, mins = divmod(mins, 60)
print("用时：%02d:%02d:%02d"% (hours, mins, secs))

Train on 2904 samples
Epoch 1/200
2904/2904 [==============================] - 5s 2ms/sample - loss: 8.3581 - acc: 0.1092
Epoch 2/200
2904/2904 [==============================] - 4s 1ms/sample - loss: 4.4483 - acc: 0.1618
Epoch 3/200
2904/2904 [==============================] - 4s 2ms/sample - loss: 3.7165 - acc: 0.1832
Epoch 4/200
2904/2904 [==============================] - 4s 2ms/sample - loss: 3.3524 - acc: 0.2266
Epoch 5/200
2904/2904 [==============================] - 5s 2ms/sample - loss: 3.1800 - acc: 0.2373
Epoch 6/200
2904/2904 [==============================] - 5s 2ms/sample - loss: 3.0554 - acc: 0.2452
Epoch 7/200
2904/2904 [==============================] - 5s 2ms/sample - loss: 2.9608 - acc: 0.2552
Epoch 8/200
2904/2904 [==============================] - 5s 2ms/sample - loss: 2.8363 - acc: 0.2703 1s 
Epoch 9/200
2904/2904 [==============================] - 5s 2ms/sample - loss: 2.7804 - acc: 0.2927
Epoch 10/200
2904/2904 [==============================] - 5s 2ms/sample - 

2904/2904 [==============================] - 3s 1ms/sample - loss: 0.9911 - acc: 0.7135
Epoch 160/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 0.9496 - acc: 0.7194
Epoch 161/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 1.0167 - acc: 0.7032
Epoch 162/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 0.9946 - acc: 0.7152
Epoch 163/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 0.9603 - acc: 0.7214
Epoch 164/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 0.9801 - acc: 0.7107
Epoch 165/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 1.0356 - acc: 0.6949
Epoch 166/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 0.9640 - acc: 0.7183
Epoch 167/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 0.9253 - acc: 0.7204
Epoch 168/200
2904/2904 [==============================] - 3s 1ms/sample - loss: 1.0168 - acc: 0

In [ ]:
#plt.plot(range(e),history.history.get('acc'))

In [108]:
#plt.plot(range(e),history.history.get('loss'))

### 储存model

In [94]:
model.save('2020.5.20.model')

INFO:tensorflow:Assets written to: 2020.5.20.model\assets


### 加载model

In [14]:
model = load_model('2020.4.3.model')

OSError: Unable to open file (unable to open file: name = '2020.4.3.model', errno = 13, error message = 'Permission denied', flags = 0, o_flags = 0)

### 清理下内存，不然不够用

In [95]:
gc.collect()

20

In [96]:
def predict(file):
    mylist = []
    index=0
    for chunk in  pd.read_csv(file, chunksize=20000):
        print(index)
        rchunk = pd.DataFrame(model.predict_classes(chunk,verbose=1))
        mylist.append(rchunk)
        index+=1
    temp_df = pd.concat(mylist, axis= 0)
    del mylist
    return temp_df

start = time.process_time()
f = open('testdata20200512.csv')
prediction = predict(f)
np.savetxt("testresult20200520.csv", prediction, fmt="%d",delimiter=",") 
elapsed = (time.process_time() - start)
print("耗时:",time.strftime("%H:%M:%S",elapsed))

0
20000/20000 [==============================] - 1s 58us/sample
1
20000/20000 [==============================] - 1s 49us/sample
2
20000/20000 [==============================] - 1s 50us/sample
3
20000/20000 [==============================] - 1s 50us/sample
4
20000/20000 [==============================] - 1s 52us/sample
5
20000/20000 [==============================] - 1s 52us/sample
6
20000/20000 [==============================] - 1s 54us/sample
7
20000/20000 [==============================] - 1s 54us/sample
8
20000/20000 [==============================] - 1s 52us/sample
9
20000/20000 [==============================] - 1s 54us/sample
10
20000/20000 [==============================] - 1s 54us/sample
11
20000/20000 [==============================] - 1s 54us/sample
12
20000/20000 [==============================] - 1s 55us/sample
13
20000/20000 [==============================] - 1s 55us/sample
14
20000/20000 [==============================] - 1s 56us/sample
15
20000/20000 [===================

TypeError: Tuple or struct_time argument required

In [98]:
prediction=model.predict_classes(train_x,verbose=1)
np.savetxt("testresult20200520_1.csv", prediction, fmt="%d",delimiter=",") 

2904/2904 [==============================] - 0s 65us/sample


In [38]:
gc.collect()
dataresult=pd.read_csv('./testdata20200403_split_7/testdata20200403_1.csv',header=0)
print("part1")
prediction = model.predict_classes(dataresult,verbose=1)
np.savetxt("./testresult20200410/testresult20200410_1.csv", prediction, fmt="%d",delimiter=",") 
gc.collect()
dataresult=pd.read_csv('./testdata20200403_split_7/testdata20200403_2.csv',header=0)
print("part2")
prediction = model.predict_classes(dataresult,verbose=1)
np.savetxt("./testresult20200410/testresult20200410_2.csv", prediction, fmt="%d",delimiter=",") 
gc.collect()
dataresult=pd.read_csv('./testdata20200403_split_7/testdata20200403_3.csv',header=0)
print("part3")
prediction = model.predict_classes(dataresult,verbose=1)
np.savetxt("./testresult20200410/testresult20200410_3.csv", prediction, fmt="%d",delimiter=",") 
gc.collect()
dataresult=pd.read_csv('./testdata20200403_split_7/testdata20200403_4.csv',header=0)
print("part4")
prediction = model.predict_classes(dataresult,verbose=1)
np.savetxt("./testresult20200410/testresult20200410_4.csv", prediction, fmt="%d",delimiter=",") 
gc.collect()
dataresult=pd.read_csv('./testdata20200403_split_7/testdata20200403_5.csv',header=0)
print("part5")
prediction = model.predict_classes(dataresult,verbose=1)
np.savetxt("./testresult20200410/testresult20200410_5.csv", prediction, fmt="%d",delimiter=",") 

KeyboardInterrupt: 